# Visualisierung


In [ ]:
# Hilfsfunktionen laden
import sys
sys.path.append('..')
import matplotlib.pyplot as plt
from kurs_helpers import (
    lade_bereinigten_text,
    GERMAN_STOPWORDS,
    GRIMM_SENTIMENT_LEXIKON,
    berechne_sentiment_mit_negation,
    pruefe_09_aufgabe_1,
    pruefe_09_aufgabe_2,
)


Durch die Berechnung des Sentiment Scores konnten wir in den vorherigen Kapiteln eine Bewertung der Stimmung eines Märchens vornehmen und diese mit anderen Märchen vergleichen.

Aber bisher haben wir immer nur Zahlen gesehen. In diesem Kapitel lernen wir zwei Arten der Visualisierung kennen:

1. **Spannungskurve** — Wie verändert sich die Stimmung *innerhalb* eines Märchens?
2. **Balkendiagramm** — Wie unterscheiden sich die Scores *zwischen* verschiedenen Märchen?

---

## Teil 1: Die Spannungskurve (Stimmungsverlauf)

Bisher haben wir immer einen einzigen Score für das gesamte Märchen berechnet. Dabei geht viel Information verloren: Ein Märchen kann düster anfangen und fröhlich enden — oder umgekehrt.

Die Idee: Wir teilen den Text in gleich große Abschnitte ein und berechnen für jeden Abschnitt einen eigenen Sentiment Score.

### Schritt 1: Text laden und Abschnittsgröße festlegen


In [ ]:
# Text laden
bereinigter_text_frauholle, wort_liste = lade_bereinigten_text('FrauHolle.txt')
print("Frau Holle enthält", len(wort_liste), "Wörter.")

# Abschnittsgröße festlegen
abschnitts_groesse = 50


### Schritt 2: Sentiment pro Abschnitt berechnen

Wir gehen Wort für Wort durch den Text. Für jeweils 50 Wörter berechnen wir einen Sentiment Score und speichern ihn in einer Liste namens `verlauf`.


In [ ]:
verlauf = []
aktueller_abschnitt_score = 0
wort_counter = 0

for wort in wort_liste:
    if wort in GRIMM_SENTIMENT_LEXIKON:
        aktueller_abschnitt_score = aktueller_abschnitt_score + GRIMM_SENTIMENT_LEXIKON[wort]
    wort_counter = wort_counter + 1

    # Wenn 50 Wörter voll sind: Score speichern und zurücksetzen
    if wort_counter == abschnitts_groesse:
        verlauf.append(aktueller_abschnitt_score)
        aktueller_abschnitt_score = 0
        wort_counter = 0

# Den letzten (unvollständigen) Abschnitt nicht vergessen
if wort_counter > 0:
    verlauf.append(aktueller_abschnitt_score)

print("Text wurde in ", len(verlauf), "Abschnitte eingeteilt.")
print("Scores pro Abschnitt: ", verlauf)


Jeder Wert in der Liste `verlauf` steht für die Stimmung eines Abschnitts:
- Positive Werte = mehr positive als negative Wörter
- Negative Werte = mehr negative als positive Wörter
- Null = ausgeglichen

### Schritt 3: Grafik erstellen

Um unsere Ergebnisse grafisch darzustellen, nutzen wir die Python-Bibliothek **matplotlib**.


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(verlauf, color="steelblue", linewidth=2)
plt.title("Stimmungsverlauf: Frau Holle", fontsize=14)
plt.xlabel("Abschnitte im Text (je 50 Wörter)")
plt.ylabel("Stimmung (Sentiment Score)")
plt.axhline(0, color="red", linestyle="--", alpha=0.5, label="Nulllinie")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Wie liest man diese Grafik?

- Die **x-Achse** zeigt die Abschnitte von links (Anfang) nach rechts (Ende).
- Die **y-Achse** zeigt die Stimmung: Werte über der roten Nulllinie sind positiv, Werte darunter negativ.

### Schritt 4: Die Grafik verbessern

Wir markieren die Flächen farbig — grün für positive, rot für negative Abschnitte:


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(verlauf, color="steelblue", linewidth=2)

# Positive Abschnitte grün markieren
positive_bereiche = []
for v in verlauf:
    positive_bereiche.append(v > 0)
plt.fill_between(range(len(verlauf)), verlauf, 0,
                 where=positive_bereiche,
                 color="green", alpha=0.2, label="positiv")

# Negative Abschnitte rot markieren
negative_bereiche = []
for v in verlauf:
    negative_bereiche.append(v < 0)
plt.fill_between(range(len(verlauf)), verlauf, 0,
                 where=negative_bereiche,
                 color="red", alpha=0.2, label="negativ")

plt.title("Stimmungsverlauf: Frau Holle", fontsize=14)
plt.xlabel("Abschnitte im Text (je 50 Wörter)")
plt.ylabel("Stimmung (Sentiment Score)")
plt.axhline(0, color="gray", linestyle="--", alpha=0.5)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Schritt 5: Verschiedene Abschnittsgrößen ausprobieren

Die Abschnittsgröße beeinflusst, wie detailliert die Kurve ist:
- **Kleine Abschnitte** (z.B. 20 Wörter): Mehr Schwankungen, detaillierter
- **Große Abschnitte** (z.B. 100 Wörter): Glattere Kurve, zeigt den Gesamttrend


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, groesse in enumerate([20, 50, 100]):
    verlauf_temp = []
    score_temp = 0
    counter_temp = 0

    for wort in wort_liste:
        if wort in GRIMM_SENTIMENT_LEXIKON:
            score_temp = score_temp + GRIMM_SENTIMENT_LEXIKON[wort]
        counter_temp = counter_temp + 1
        if counter_temp == groesse:
            verlauf_temp.append(score_temp)
            score_temp = 0
            counter_temp = 0
    if counter_temp > 0:
        verlauf_temp.append(score_temp)

    axes[idx].plot(verlauf_temp, color="steelblue", linewidth=2)
    axes[idx].axhline(0, color="red", linestyle="--", alpha=0.5)
    axes[idx].set_title("Abschnittsgröße: ", groesse, "Wörter")
    axes[idx].set_xlabel("Abschnitte")
    axes[idx].set_ylabel("Sentiment")
    axes[idx].grid(True, alpha=0.3)

plt.suptitle("Frau Holle — Stimmungsverlauf bei verschiedenen Abschnittsgrößen", fontsize=14)
plt.tight_layout()
plt.show()


---

## Teil 2: Balkendiagramm — Märchen vergleichen

Die Spannungskurve zeigt den Verlauf *innerhalb* eines Märchens. Aber wie vergleicht man die Stimmung *zwischen* verschiedenen Märchen? Dafür eignet sich ein **Balkendiagramm**.

Berechnen wir die Sentiment Scores für drei bekannte Märchen:


In [ ]:
# Drei Märchen laden und analysieren
maerchen_dateien = {
    "Frau Holle": "FrauHolle.txt",
    "Schneewittchen": "Sneewittchen.txt",
    "Hänsel & Gretel": "HaenselGrethel.txt",
}

namen = []
scores = []

for titel, datei in maerchen_dateien.items():
    _, wl = lade_bereinigten_text(datei)

    # Stoppwörter und Zahlen entfernen
    gefiltert = []
    for wort in wl:
        if wort not in GERMAN_STOPWORDS and not wort.isdigit():
            gefiltert.append(wort)

    score, _ = berechne_sentiment_mit_negation(gefiltert)
    namen.append(titel)
    scores.append(score)
    print(f"{titel}: Score = {score}")


Jetzt stellen wir die Scores als Balkendiagramm dar:


In [ ]:
# Farben: grün für positiv, rot für negativ
farben = []
for s in scores:
    if s > 0:
        farben.append("green")
    elif s < 0:
        farben.append("red")
    else:
        farben.append("gray")

plt.figure(figsize=(8, 5))
plt.bar(namen, scores, color=farben, alpha=0.7, edgecolor="black")
plt.title("Sentiment Score — Märchenvergleich", fontsize=14)
plt.ylabel("Sentiment Score")
plt.axhline(0, color="gray", linestyle="--", alpha=0.5)
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


Grüne Balken zeigen Märchen mit positiver Stimmung, rote Balken Märchen mit negativer Stimmung.

### Erweiterung: Positive und negative Wörter aufschlüsseln


In [ ]:
import numpy as np
positive_counts = []
negative_counts = []

for titel, datei in maerchen_dateien.items():
    bereinigter_text_temp, wl = lade_bereinigten_text(datei)
    gefiltert = []
    for wort in wl:
        if wort not in GERMAN_STOPWORDS and not wort.isdigit():
            gefiltert.append(wort)
    score_temp, details = berechne_sentiment_mit_negation(gefiltert)
    pos = 0
    neg = 0
    for wort, orig, negiert, end in details:
        if end > 0:
            pos = pos + 1
        elif end < 0:
            neg = neg + 1
    positive_counts.append(pos)
    negative_counts.append(neg)

x = np.arange(len(namen))
breite = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - breite/2, positive_counts, breite, label="Positive Wörter", color="green", alpha=0.7)
plt.bar(x + breite/2, negative_counts, breite, label="Negative Wörter", color="red", alpha=0.7)
plt.xticks(x, namen)
plt.title("Positive vs. negative Wörter — Märchenvergleich", fontsize=14)
plt.ylabel("Anzahl Wörter")
plt.legend()
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


---

## Übungen

### Aufgabe 1: Spannungskurve

Erstellen Sie eine Stimmungsverlaufsgrafik für ein Märchen Ihrer Wahl. Speichern Sie den Verlauf in `verlauf_aufgabe` und den Märchennamen in `maerchen_name_aufgabe`.


In [ ]:
# Wählen Sie ein Märchen:
maerchen_name_aufgabe = "..."  # z.B. "Dornroeschen.txt"

# Text laden

# Abschnittsgröße festlegen

# Sentiment pro Abschnitt berechnen
verlauf_aufgabe = []

# Grafik erstellen


In [ ]:
# Überprüfung:
pruefe_09_aufgabe_1()


### Aufgabe 2: Balkendiagramm

Wählen Sie drei weitere Märchen und erstellen Sie ein Balkendiagramm. Speichern Sie die Namen in `namen_aufgabe` und die Scores in `scores_aufgabe`.


In [ ]:
# Wählen Sie drei Märchen:
maerchen_auswahl = {
    "...": "....txt",
    "...": "....txt",
    "...": "....txt",
}

namen_aufgabe = []
scores_aufgabe = []

# Scores berechnen

# Balkendiagramm erstellen


In [ ]:
# Überprüfung:
pruefe_09_aufgabe_2()


---

**Geschafft!** Sie können jetzt Sentimentanalysen nicht nur berechnen, sondern auch visualisieren.

[← Zurück zur Übersicht](../00_Willkommen.ipynb) | [← Zurück zu Kapitel 8: Sentimentanalyse vertiefen](../Teil_2/08_Sentimentanalyse_vertiefen.ipynb) | [Weiter zu Kapitel 10: Abschlussprüfung Teil II →](10_Abschlusspruefung_Teil_II.ipynb)
